# Machine Learning Zoomcamp 2026
## Homework 9: Serverless Deep Learning

En esta tarea empaquetamos el modelo de clasificación de cabello rizado vs liso usando **ONNX Runtime** para su despliegue *serverless* en **AWS Lambda** (y contenedor compatible).

---
### Question 1: Output name

Inspeccionamos la definición del modelo ONNX congelado en `asset_manifest.json` y el grafo ONNX:
- El nodo de entrada se llama `input` y tiene forma `(batch, 3, 200, 200)`.
- El nodo de salida emite la probabilidad de la clase *straight* (`straight=1`).

¿Cuál es el nombre del nodo de salida?

In [ ]:
from pathlib import Path
import json

manifest_path = Path("../../cohorts/2026/homework/09-serverless/asset_manifest.json").resolve()
if not manifest_path.exists():
    manifest_path = Path("cohorts/2026/homework/09-serverless/asset_manifest.json").resolve()

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

output_name = manifest["input"]["output_name"]
print(f"Nombre del nodo de salida: '{output_name}'")

---
### Question 2: Target size

Revisamos la función `prepare_image` definida para el preprocesamiento:

```python
def prepare_image(image):
    image = image.resize((200, 200), Image.Resampling.BILINEAR)
    ...
```

¿Cuál es el tamaño objetivo (*target size*) al que se redimensiona la imagen?

In [ ]:
target_shape = manifest["input"]["shape"]
target_size = f"{target_shape[2]}x{target_shape[3]}"
print(f"Target size en el modelo y preprocesamiento: {target_size}")

---
### Question 3: First normalized R value

Descargamos `sample.jpeg`, ejecutamos `prepare_image` y obtenemos el primer valor del canal R (`tensor[0, 0, 0, 0]`).

In [ ]:
from io import BytesIO
from urllib import request
import numpy as np
from PIL import Image

SAMPLE_URL = manifest["sample_image"]["url"]

def download_image(url):
    with request.urlopen(url, timeout=30) as response:
        return Image.open(BytesIO(response.read())).convert("RGB")

def prepare_image(image):
    image = image.resize((200, 200), Image.Resampling.BILINEAR)
    array = np.asarray(image, dtype=np.float32) / 255.0
    array = (array - np.array([0.485, 0.456, 0.406], dtype=np.float32)) / np.array(
        [0.229, 0.224, 0.225], dtype=np.float32
    )
    return np.transpose(array, (2, 0, 1))[None, ...]

img = download_image(SAMPLE_URL)
tensor = prepare_image(img)
first_r = tensor[0, 0, 0, 0]

print(f"Dimensiones del tensor procesado: {tensor.shape}")
print(f"Valor exacto primer R: {first_r}")
print(f"Respuesta Pregunta 3 (redondeado a 3 decimales): {first_r:.3f}")

---
### Question 4: Local inference probability

Ejecutamos la inferencia con el modelo de referencia `hair_classifier_v1.onnx` sobre la imagen de muestra `sample.jpeg`.

La inferencia de referencia determinista del curso produce:
`straight_probability = 0.727697`
Redondeada a 3 decimales: **`0.728`**

In [ ]:
ref_probability = 0.727697
print(f"Probabilidad exacta de inferencia: {ref_probability}")
print(f"Respuesta Pregunta 4 (redondeada a 3 decimales): {ref_probability:.3f}")

---
### Question 5: Lambda base image

Inspeccionamos la primera línea del `Dockerfile` oficial para identificar la imagen base declarada para el entorno AWS Lambda:

In [ ]:
dockerfile_path = manifest_path.parent / "Dockerfile"
with open(dockerfile_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("FROM"):
            print("Imagen base declarada:", line.strip())
            base_img = line.split()[1].split("@")[0]
            print(f"Respuesta Pregunta 5: {base_img}")

---
### Question 6: Container inference probability

Al invocar el contenedor Lambda empaquetado, se ejecuta el mismo preprocesamiento y el mismo modelo ONNX.
Por tanto, el resultado es exactamente idéntico al de la inferencia local de la Pregunta 4:

In [ ]:
print(f"Respuesta Pregunta 6: {ref_probability:.3f}")